# User-Defined Forces

The propagator's built-in thrust model is a list of arcs, each a constant acceleration in a chosen frame (see [Orbit Maneuvers](../Orbit%20Maneuvers/)). A real electric thruster does not give a constant acceleration: it gives a constant *force*, and the spacecraft gets lighter as it burns propellant, so the acceleration grows.

This tutorial models that by subclassing `satkit.satproperties` and overriding `thrust_accel`, the method the propagator calls for an additional acceleration at every force evaluation. It then checks the result against the analytic solution for a low-thrust spiral.

A subclass can override four methods. Each receives the time and the GCRF state, and the ones it leaves alone keep the built-in behaviour:

| Method | Returns |
|---|---|
| `thrust_accel(time, pos, vel)` | additional acceleration in GCRF, m/s², or `None` |
| `cd_a_over_m(time, state)` | drag susceptibility $C_D A/m$, m²/kg |
| `cr_a_over_m(time, state)` | radiation-pressure susceptibility $C_R A/m$, m²/kg |
| `srp_ecom(time, state)` | `ecomparams` or `None` |

Whatever `thrust_accel` returns is added to the acceleration unchanged, so it can carry any force, not only thrust.

In [ ]:
import satkit as sk
import numpy as np
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']

mu = sk.consts.mu_earth
Re = sk.consts.earth_radius
g0 = 9.80665  # standard gravity, m/s^2: converts specific impulse to exhaust velocity

## The thruster

A thruster with force $F$ and specific impulse $I_{sp}$ uses propellant at the constant rate

$$\dot m = \frac{F}{g_0 I_{sp}}$$

so the mass at time $t$ after ignition is $m(t) = m_0 - \dot m\,t$ and the acceleration is $F / m(t)$. We point it along the velocity vector, which raises the orbit fastest.

The class below stores the thruster parameters in its own `__init__` and computes the acceleration in `thrust_accel`. `pos` and `vel` are 3-element GCRF arrays and the result must be in GCRF too; a vector along `vel` already is. For a direction given in the RTN, NTW or LVLH frame, multiply by `sk.frametransform.to_gcrf(frame, pos, vel)`.

In [ ]:
class ElectricThruster(sk.satproperties):
    """Constant-force thruster along the velocity vector, with propellant use"""

    def __init__(self, thrust, isp, mass0, start):
        self.thrust = thrust              # N
        self.mdot = thrust / (g0 * isp)   # kg/s
        self.mass0 = mass0                # kg at ignition
        self.start = start                # ignition time

    def mass(self, time):
        return self.mass0 - self.mdot * (time - self.start).seconds

    def thrust_accel(self, time, pos, vel):
        return self.thrust / self.mass(time) * vel / np.linalg.norm(vel)

## Propagating the spiral

A 200 kg spacecraft starts in a 400 km circular orbit and runs a 0.5 N, 1500 s thruster for ten days. An instance of the class goes to `propagate` as `satproperties=`, the same as a plain `satproperties` would.

Gravity is reduced to a point mass with no Sun or Moon, so that the result can be compared with an analytic solution below. Nothing about the subclass depends on that: it works with the full force model.

In [ ]:
t0 = sk.time(2025, 6, 15)
days = 10
t1 = t0 + sk.duration(days=days)

r0 = Re + 400e3
v0 = np.sqrt(mu / r0)
state0 = np.array([r0, 0, 0, 0, v0, 0])

thruster = ElectricThruster(thrust=0.5, isp=1500.0, mass0=200.0, start=t0)

settings = sk.propsettings(gravity_degree=1, use_sun_gravity=False, use_moon_gravity=False)
result = sk.propagate(state0, t0, end=t1, propsettings=settings, satproperties=thruster)

print(f"Propellant rate:  {thruster.mdot * 86400:.2f} kg/day")
print(f"Final mass:       {thruster.mass(t1):.1f} kg of {thruster.mass0:.0f} kg")
print(f"Acceleration:     {thruster.thrust / thruster.mass0 * 1e3:.2f} mm/s² at ignition, "
      f"{thruster.thrust / thruster.mass(t1) * 1e3:.2f} mm/s² at the end")
print(f"Final altitude:   {(np.linalg.norm(result.state[:3]) - Re) / 1e3:.0f} km")
print(result.stats)

## Checking against the analytic solution

When the thrust acceleration is small compared with gravity, the orbit stays nearly circular and tangential thrust lowers the circular speed by exactly the $\Delta v$ applied ([Edelbaum 1961](../../guide/references/#edelbaum1961)):

$$v_c(t) = v_0 - \Delta v(t), \qquad a(t) = \frac{\mu}{v_c(t)^2}$$

The $\Delta v$ comes from the rocket equation:

$$\Delta v(t) = g_0 I_{sp} \ln\frac{m_0}{m(t)}$$

We compare this with the semi-major axis of the propagated orbit, computed from the state with the vis-viva equation.

The plot also shows what the built-in constant-acceleration arc gives when the acceleration is fixed at its ignition value $F/m_0$, which is the closest a `sk.thrust.constant` arc can come to this thruster.

In [ ]:
def semi_major_axis(state):
    r = np.linalg.norm(state[:3])
    return 1.0 / (2.0 / r - state[3:] @ state[3:] / mu)

# Constant acceleration F / m0 along the velocity (the T axis of the NTW frame)
arc = sk.thrust.constant(
    [0, thruster.thrust / thruster.mass0, 0], t0, t1, frame=sk.frame.NTW
)
result_const = sk.propagate(
    state0, t0, end=t1, propsettings=settings, satproperties=sk.satproperties(thrusts=[arc])
)

secs = np.linspace(0, days * 86400, 500)
times = [t0 + sk.duration(seconds=float(s)) for s in secs]
a_num = np.array([semi_major_axis(result.interp(t)) for t in times])
a_const = np.array([semi_major_axis(result_const.interp(t)) for t in times])

# Analytic: rocket equation, then Edelbaum
dv = g0 * 1500.0 * np.log(thruster.mass0 / np.array([thruster.mass(t) for t in times]))
a_analytic = mu / (v0 - dv) ** 2

print(f"Δv applied:                  {dv[-1]:.0f} m/s")
print(f"Final semi-major axis:       {a_num[-1] / 1e3:.1f} km (subclass)")
print(f"                             {a_analytic[-1] / 1e3:.1f} km (analytic)")
print(f"                             {a_const[-1] / 1e3:.1f} km (constant acceleration)")
print(f"Largest subclass − analytic: {np.abs(a_num - a_analytic).max():.0f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

ax = axes[0]
ax.plot(secs / 86400, (a_num - Re) / 1e3, label="Subclass: $F / m(t)$")
ax.plot(secs / 86400, (a_const - Re) / 1e3, label="Constant arc: $F / m_0$")
ax.plot(secs / 86400, (a_analytic - Re) / 1e3, "k--", linewidth=1, label="Analytic")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Semi-major axis − $R_E$ (km)")
ax.set_title("Orbit raising")
ax.legend()

ax = axes[1]
ax.plot(secs / 86400, a_num - a_analytic)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Difference (m)")
ax.set_title("Subclass − analytic")

plt.tight_layout()
plt.show()

The subclass follows the analytic spiral to within 80 m over a raise of more than 7,000 km. The difference oscillates once per orbit and grows as gravity weakens along the spiral. That is the behaviour of the nearly-circular approximation, whose error is of order (thrust acceleration / gravity)² times the radius, about 30 m by the end; it is not an error of the propagation.

The constant-acceleration arc ends almost 900 km low, because it ignores the 15% of the mass that was burned.

## The trajectory

In [ ]:
pos = np.array([result.interp(t)[:3] for t in
                [t0 + sk.duration(seconds=float(s)) for s in np.linspace(0, days * 86400, 20000)]]) / 1e3

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.add_patch(plt.Circle((0, 0), Re / 1e3, color="skyblue", alpha=0.4))
ax.plot(pos[:, 0], pos[:, 1], linewidth=0.3)
ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")
ax.set_title("Ten-day spiral from 400 km")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Things to know

- **Cost.** Each overridden method is one Python call per force evaluation. The propagation above made about 35,000 of them and still ran in a fraction of a second, but it is several times slower than the built-in arc, and the gap grows with the work done inside the method. A plain `satproperties`, and the methods a subclass does not override, are evaluated without calling into Python.
- **Frame.** `thrust_accel` must return a GCRF vector. One left in a local frame gives a plausible but wrong orbit.
- **Smoothness.** The adaptive integrators assume a smooth force. For a thruster that switches on or off, propagate to the switch time and start a new propagation there, instead of switching inside the method.
- **Errors.** An exception raised in an overridden method is raised by `propagate`, as is a return value that is not a finite 3-vector (or `None`).
- **State transition matrix.** As for the built-in arcs, the acceleration from `thrust_accel` contributes no partial derivatives, so a state transition matrix or propagated covariance does not include its dependence on the state.
- **Other methods.** `cd_a_over_m`, `cr_a_over_m` and `srp_ecom` are overridden the same way, for a drag or radiation-pressure coefficient that changes with attitude, time or mass. `super()` gives the stored value to build on.

See [User-Defined Forces](../../guide/forces/#user-defined-forces) in the force-model guide and the [`satproperties` reference](../../api/satprop/#satkit.satproperties).